# MCP: the tool protocol

Booster LEAP, Agentic AI, topic 2. Run a real MCP server object, look at what a client sees, and read the messages on the wire.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use this notebook:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again; the experiments tell you what to try.

## Stage 1: a real server object

MCP (Model Context Protocol) lets a tool be written once as a server and used by any client. The Python package is called `mcp`. This notebook uses version 2.x, where the class is `MCPServer` (in version 1 it was called `FastMCP`, so old tutorials do not match). The server below lives inside the notebook, so no network and no key are needed.

In [ ]:
%pip -q install "mcp>=2"

In [ ]:
from mcp.server.mcpserver import MCPServer

server = MCPServer("support-desk")


@server.tool()
def lookup_order(order_id: str) -> str:
    """Find one order by its id and return its status."""
    return {"A1001": "shipped", "A1002": "delivered"}.get(order_id, "no such order")

A server lists its tools (`tools/list`) and runs one on request (`tools/call`). The cell asks the server object directly, the way a client would after connecting.

In [ ]:
# In a notebook a loop is already running, so we use await directly (asyncio.run would fail here).
for tool in await server.list_tools():            # what a client sees when it sends tools/list
    schema = tool.input_schema
    print(tool.name, "|", tool.description, "|", schema["properties"], "required:", schema["required"])
result = await server.call_tool("lookup_order", {"order_id": "A1001"})   # what tools/call returns
print("call ->", result.content[0].text, "| is_error:", result.is_error)

lookup_order | Find one order by its id and return its status. | {'order_id': {'title': 'Order Id', 'type': 'string'}} required: ['order_id']
call -> shipped | is_error: False


## Stage 2: the same conversation as messages

The toy server below speaks MCP-shaped JSON-RPC messages, so you can see what travels between a client and a server. It has the same tool and answers four requests: list, a successful call, a failed call and an unknown method.

In [ ]:
import json

# A toy server that speaks MCP-shaped messages (JSON-RPC 2.0). It shows the shape of the conversation; the real protocol has more messages.
ORDERS = {"A1001": "shipped", "A1002": "delivered"}
TOOLS = [{"name": "lookup_order", "description": "Find one order by its id and return its status.",
          "inputSchema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}]

def handle(request):
    method, params = request["method"], request.get("params", {})
    if method == "tools/list":
        result = {"tools": TOOLS}
    elif method == "tools/call" and params.get("name") == "lookup_order":
        status = ORDERS.get(params["arguments"].get("order_id"))
        text = f"order status: {status}" if status else "no such order"
        result = {"content": [{"type": "text", "text": text}], "isError": status is None}
    else:
        return {"jsonrpc": "2.0", "id": request["id"], "error": {"code": -32601, "message": f"unknown method or tool: {method}"}}
    return {"jsonrpc": "2.0", "id": request["id"], "result": result}

requests = [
    {"jsonrpc": "2.0", "id": 1, "method": "tools/list"},
    {"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": "lookup_order", "arguments": {"order_id": "A1001"}}},
    {"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "lookup_order", "arguments": {"order_id": "Z9"}}},
    {"jsonrpc": "2.0", "id": 4, "method": "tools/destroy"},
]
for r in requests:
    print(">>", json.dumps(r))
    print("<<", json.dumps(handle(r)))

>> {"jsonrpc": "2.0", "id": 1, "method": "tools/list"}
<< {"jsonrpc": "2.0", "id": 1, "result": {"tools": [{"name": "lookup_order", "description": "Find one order by its id and return its status.", "inputSchema": {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]}}]}}
>> {"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {"name": "lookup_order", "arguments": {"order_id": "A1001"}}}
<< {"jsonrpc": "2.0", "id": 2, "result": {"content": [{"type": "text", "text": "order status: shipped"}], "isError": false}}
>> {"jsonrpc": "2.0", "id": 3, "method": "tools/call", "params": {"name": "lookup_order", "arguments": {"order_id": "Z9"}}}
<< {"jsonrpc": "2.0", "id": 3, "result": {"content": [{"type": "text", "text": "no such order"}], "isError": true}}
>> {"jsonrpc": "2.0", "id": 4, "method": "tools/destroy"}
<< {"jsonrpc": "2.0", "id": 4, "error": {"code": -32601, "message": "unknown method or tool: tools/destroy"}}


## Experiments

1. In stage 1 add a second tool `ping` that returns `"pong"`. How many tools does `tools/list` show now?
2. Add a parameter `verbose: bool = False` to `lookup_order`. How does the input schema change, and is `verbose` required?
3. Call the tool with `{"order_id": 5}`. What happens, and where would you validate this in your own agent?
4. In stage 2 add a `tools/call` request for a tool that does not exist. Is the answer a result with `isError` or a protocol error? Why?
5. Read the description of the tool as the model would. Could a server author word a description to steer the model? What would you check before adding a third-party server?